In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

# Retrieve widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

# print(f"source_system : {source_system}")
# print(f"table_name    : {table_name}")
# print(f"timestamp     : {timestamp}")

In [0]:
#/Volumes/dbr_caresync/landing/landing_volume/neon_postgres/hospitals/2026-10-01_06:28:19/
path = f'/Volumes/dbr_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

try:
    dbutils.fs.ls(path)
except Exception:
    print("no new files are present")
    dbutils.notebook.exit("no new files are present")

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(path)
else:
    landing_df = spark.read.format("parquet").load(path)

display(landing_df)

In [0]:
%skip
landing_df= spark.read.format("parquet").load(path)
display(landing_df)

In [0]:
from pyspark.sql import functions as F

updated_df = landing_df \
    .withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())

#display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"dbr_caresync.bronze.{table_name}")

In [0]:
%skip
%sql
select * from dbr_caresync.bronze.hospitals